# v43 - ROI experts: Lateral and Medial Meniscus on the v42 crops (fold 0, local GPU)

Design: `docs/research/v42-roi-experts-design.md` revision 2, section 6 (Codex-reviewed; decision D-031), with the box
parameters frozen by pilot 3 (section 11). Two single-finding models: **LM** (Lateral Meniscus, lateral crops) and **MM**
(Medial Meniscus, medial crops). They are not submitted alone: `v44-submit` blends each into its own column of the incumbent
by the order-only rule of design section 5; the other ten columns stay byte-equal.

**Input.** The `lingxd/v42-roi-cache` output (version 5): for every training study and compartment, the sagittal slices
within +-14 mm of the compartment centre (joint centre +-18 mm along the patient left-right axis), each resampled to a box of
AP 72 x SI 40 mm at 0.25 mm/px (160 x 288 px, fixed physical axes), uint8, in physical order. Only studies whose box passes
the v42 usability rule (SKM-TEA joint anchor, a single side, >= 3 slices, >= 80% of the box inside the image) are used.

**Model.** Up to 16 windows per study, each three adjacent slices as channels (all centres when a study has <= 16 slices);
`convnext_nano.in12k_ft_in1k`; the v36 gated-attention pooling with one output. Target: arm P's round-3 target for the
finding (`0.5 x v06 soft + 0.5 x v13 arm-N OOF`; gold excluded); v06 loss weights. Fold 0 only.

**Training.** AdamW (backbone 1e-4, head 1e-3, weight decay 0.05), one warm-up epoch then cosine, effective batch 16 studies,
fp16 (micro-batches of 8 studies, windows padded to a multiple of 16 so that only a few batch shapes occur), the v36
augmentation (rotation, scale, shift, gamma, brightness; no flips), at most 20 epochs with a patience of 6
epochs; the checkpoint with the best validation AUC of the finding (against `soft >= 0.5`, usable fold-0 rows) is kept.

**Diagnostics only** (no gate, no selection): on fold 0, AUC of the expert, of arm N's OOF and of their rank mean on the
usable rows, and on all fold-0 rows arm N's OOF against the v44 order-only blend (expert on usable rows, N elsewhere), with a
paired bootstrap; the same on the 58 gold studies with arm N's fold-0 gold predictions. Fold-0 validation is leaky: the
training targets of folds 1-4 contain N OOF from models that saw fold 0 (v36 design section 3).

## Provenance

Derived from our own `notebooks/v36-gap-diagnostics.ipynb` (trainer: augmentation, gated-attention head, optimiser, schedule,
memory probe, weighted BCE). Changes: the ROI crops as input (one compartment, rectangular 160 x 288, windows of adjacent
slices), one output, the backbone and learning rates above, patience, no 5-slot structure. The ROI crops come from
`notebooks/v42-roi-cache.ipynb` (heliosli's public S6 code path for geometry, side and the SKM-TEA joint anchor; D-023,
D-031). The per-finding ROI-expert idea comes from a public write-up by another team (github.com/VaryaStrizhneva/RSNA-project);
none of its code, annotations or weights are used.

Runs on the local GPU (D-016): `python scripts/local_v13.py --version v43 experts` in the `kaggle-gpu` env. Kaggle input
names: `v06-dicom-audit` (v06_targets.csv, v06_folds.csv), `rsna-knee-v13n-oof` (v13 arm-N OOF), `v42-roi-cache`
(index_0.csv, index_1.csv, crops/), optionally the v13 arm-N fold-0 gold predictions (diagnostics).

In [ ]:
# v43 configuration
import json
import os
import subprocess
import sys
import time
from pathlib import Path

VERSION = 'v43'
FOLD = 0
EXPERTS = [('LM', 'Lateral Meniscus', 'lateral'), ('MM', 'Medial Meniscus', 'medial')]   # tag, finding, v42 compartment
NOTEBOOK_T0 = time.time()
TIME_LIMIT_S = 3 * 3600        # per expert
WORK = Path('/kaggle/working') / VERSION
WORK.mkdir(parents=True, exist_ok=True)

LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA',
          'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
OOF_WEIGHT = 0.5               # arm P's target = (1 - OOF_WEIGHT) * v06 soft + OOF_WEIGHT * v13 arm-N OOF
N_OOF_FILES = {k: f'v13_oof_fold{k}_N_k16.csv' for k in range(5)}
N_GOLD_FILE = 'v13_gold_fold0_N_k16.csv'   # diagnostics only; optional

CFG = {
    'version': VERSION,
    'fold': FOLD,
    'backbone': 'convnext_nano.in12k_ft_in1k',
    'pretrained': True,
    'img_h': 160,
    'img_w': 288,
    'lr_backbone': 1e-4,
    'lr_head': 1e-3,
    'weight_decay': 0.05,
    'epochs': 20,
    'warmup_epochs': 1,
    'patience': 6,
    'eval_from_epoch': 2,
    'k_train': 16,
    'k_infer': 16,
    'effective_batch_studies': 16,
    'micro_batch_studies': 8,      # with accumulation 2; keeps the peak well inside the 12 GB of the local GPU
    'eval_batch_studies': 8,
    'num_workers': 2,
    'seed': 2026,
    'device': 'cuda',
}
print(f'{VERSION}: experts {[e[0] for e in EXPERTS]}, fold {FOLD}, {CFG["backbone"]}, '
      f'{CFG["img_h"]} x {CFG["img_w"]}, epochs <= {CFG["epochs"]}', flush=True)


In [ ]:
# Locate the inputs without walking the competition image tree.
def find_unique_file(name, root=Path('/kaggle/input'), required=True):
    hits = []
    for current, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series', 'competitions', 'cache', 'crops', 'montage')
                   and 'rsna-knee-abnormality-detection' not in d]
        if name in files:
            hits.append(Path(current) / name)
    if len(hits) != 1 and (required or hits):
        raise FileNotFoundError(f'expected one {name} under {root}, found {hits}')
    return hits[0] if hits else None

PATHS = {
    'targets': str(find_unique_file('v06_targets.csv')),
    'folds': str(find_unique_file('v06_folds.csv')),
}
ROI_DIR = find_unique_file('index_0.csv').parent
assert (ROI_DIR / 'index_1.csv').is_file() and (ROI_DIR / 'crops').is_dir(), ROI_DIR
N_OOF_PATHS = {k: find_unique_file(name) for k, name in N_OOF_FILES.items()}
N_GOLD_PATH = find_unique_file(N_GOLD_FILE, required=False)
print(json.dumps({**PATHS, 'roi_dir': str(ROI_DIR), 'n_oof': {k: str(v) for k, v in N_OOF_PATHS.items()},
                  'n_gold': str(N_GOLD_PATH) if N_GOLD_PATH else None}, indent=2))


## Training target and the usable studies

Arm P's target is rebuilt exactly as in v36 (every non-gold study with an arm-N OOF row; the local runner checks it against an
independent recomputation). The usable studies per compartment come from the v42 index; the counts per split are reported.

In [ ]:
import hashlib
import numpy as np
import pandas as pd

targets = pd.read_csv(PATHS['targets'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
folds = pd.read_csv(PATHS['folds'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
fold_of = folds.fold.reindex(targets.index)
gold_ids = set(targets.index[targets.is_gold.astype(bool)])

parts = []
for k, path in sorted(N_OOF_PATHS.items()):
    df = pd.read_csv(path, dtype={'StudyInstanceUID': str})
    assert df.columns.tolist() == ['StudyInstanceUID'] + LABELS and df.StudyInstanceUID.is_unique, path.name
    assert set(fold_of.reindex(df.StudyInstanceUID)) == {k}, f'{path.name}: rows outside fold {k}'
    parts.append(df.set_index('StudyInstanceUID'))
n_oof = pd.concat(parts)
assert n_oof.index.is_unique and not (set(n_oof.index) & gold_ids), 'N OOF rows must be unique and non-gold'
pool = [u for u in targets.index if u in n_oof.index]
soft = targets.loc[pool, LABELS].astype(np.float64)
assert np.isfinite(soft.to_numpy()).all(), 'missing soft target'
p_target = (1.0 - OOF_WEIGHT) * soft + OOF_WEIGHT * n_oof.loc[pool, LABELS].astype(np.float64)
p_target.index.name = 'StudyInstanceUID'
PATHS['train_targets'] = str(WORK / 'v43_train_targets_P.csv')
p_target.to_csv(PATHS['train_targets'])

roi = pd.concat([pd.read_csv(ROI_DIR / f'index_{w}.csv', dtype={'StudyInstanceUID': str}) for w in (0, 1)],
                ignore_index=True).set_index('StudyInstanceUID')
assert roi.index.is_unique and set(roi.index) == set(targets.index), 'the v42 index must cover every training study once'
counts = {}
for tag, finding, comp in EXPERTS:
    use = roi[f'{comp}_usable'].astype(str).str.lower().eq('true')
    rows = roi.loc[use, [f'{comp}_shard', f'{comp}_start', f'{comp}_n']].copy()
    rows.columns = ['shard', 'start', 'n']
    assert rows.shard.notna().all() and (rows.n >= 3).all() and (rows.start >= 0).all()
    assert all((ROI_DIR / 'crops' / s).is_file() for s in rows.shard.unique()), 'a shard is missing'
    rows.to_csv(WORK / f'v43_rows_{tag}.csv')
    ids = set(rows.index)
    tr = [u for u in pool if fold_of[u] != FOLD]
    va = [u for u in pool if fold_of[u] == FOLD]
    counts[tag] = {'finding': finding, 'compartment': comp,
                   'train': [sum(u in ids for u in tr), len(tr)], 'val': [sum(u in ids for u in va), len(va)],
                   'gold': [len(ids & gold_ids), len(gold_ids)],
                   'train_pos_rate_soft05': round(float((soft.loc[[u for u in tr if u in ids], finding] >= 0.5).mean()), 4),
                   'slices_median': float(rows.n.median()), 'slices_max': int(rows.n.max())}
summary = {'version': VERSION, 'n_pool': len(pool), 'n_gold_excluded': len(gold_ids), 'oof_weight': OOF_WEIGHT,
           'train_targets_sha256': hashlib.sha256(Path(PATHS['train_targets']).read_bytes()).hexdigest(),
           'n_oof_sha256': {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in N_OOF_PATHS.values()},
           'usable_counts': counts}
(WORK / 'v43_targets_summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary['usable_counts'], indent=2))


## Trainer script

Written to disk and run as its own process, one expert at a time. The notebook is the source of truth; the script is
regenerated on every run. `v44-submit` reuses `RoiStore`, `windows`, `Net` and `predict` unchanged.

In [ ]:
%%writefile /kaggle/working/v43_train.py
"""v43 ROI expert trainer: one finding, one fold, one GPU. Configuration comes as a JSON argument.

Derived from the v36 trainer (augmentation, gated-attention pooling, optimiser, schedule, memory probe, weighted BCE).
Changed for design v42-roi-experts section 6: the input is the v42 ROI crops of one compartment (rectangular 160 x 288,
up to 16 windows of three adjacent slices), one output, patience on the validation AUC, no slots and no smoke mode.
"""
import hashlib
import json
import math
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)[:, None, None]
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)[:, None, None]


def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)


# ----------------------------------------------------------------------------- data
class RoiStore:
    """Read-only access to the v42 shards: (shard, start, n) -> uint8 [n, 160, 288]; memory-mapped, opened lazily in
    each process."""

    def __init__(self, crops_dir):
        self.dir, self.arrays = Path(crops_dir), {}

    def get(self, shard, start, n):
        a = self.arrays.get(shard)
        if a is None:
            a = self.arrays[shard] = np.load(self.dir / shard, mmap_mode='r')
        assert start + n <= len(a), (shard, start, n, len(a))
        return np.asarray(a[start:start + n])


def window_centres(n, k, train, rng):
    # Up to k distinct centres over n physically ordered slices; every slice when n <= k.
    if n <= k:
        return np.arange(n)
    if train:
        return np.sort(rng.choice(n, k, replace=False))
    return np.linspace(0, n - 1, k).round().astype(int)


def augment(tri, rng):
    # tri: (3, H, W) uint8 triplet; the same geometric transform for all three channels (v36, unchanged).
    h, w = tri.shape[1:]
    angle = rng.uniform(-10, 10)
    scale = rng.uniform(0.9, 1.1)
    tx, ty = rng.uniform(-0.05, 0.05) * w, rng.uniform(-0.05, 0.05) * h
    m = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
    m[:, 2] += (tx, ty)
    img = cv2.warpAffine(np.ascontiguousarray(tri.transpose(1, 2, 0)), m, (w, h),
                         flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    img = img.astype(np.float32) / 255.0
    gamma = rng.uniform(0.85, 1.15)
    img = np.clip(img, 0, 1) ** gamma
    img = img * rng.uniform(0.9, 1.1) + rng.uniform(-0.05, 0.05)
    return np.clip(img, 0, 1).transpose(2, 0, 1)


def windows(stack, k, train, rng=None):
    # stack: uint8 [n, H, W] in physical order -> float32 [m, 3, H, W], m = min(n, k), ImageNet-normalised.
    n = len(stack)
    out = []
    for c in window_centres(n, k, train, rng):
        tri = stack[[min(max(int(c) + d, 0), n - 1) for d in (-1, 0, 1)]]
        x = augment(tri, rng) if train else tri.astype(np.float32) / 255.0
        out.append((x - MEAN) / STD)
    return np.stack(out).astype(np.float32)


class RoiDataset(Dataset):
    def __init__(self, rows, y, w, crops_dir, k, train, seed):
        # rows: list of (uid, shard, start, n); y, w: float32 arrays aligned with rows
        self.rows, self.y, self.w = list(rows), y, w
        self.crops_dir, self.k, self.train, self.seed = crops_dir, k, train, seed
        self.epoch, self.store = 0, None

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        if self.store is None:
            self.store = RoiStore(self.crops_dir)
        rng = np.random.default_rng((self.seed, self.epoch, i)) if self.train else None
        uid, shard, start, n = self.rows[i]
        x = windows(self.store.get(shard, start, n), self.k, self.train, rng)
        return {'x': x, 'y': np.float32(self.y[i]), 'w': np.float32(self.w[i]), 'uid': uid}


def collate(batch):
    return batch


# ---------------------------------------------------------------------------- model
class Net(nn.Module):
    """v36's gated-attention pooling over the windows of a study, one output."""

    def __init__(self, backbone, att_dim=256, drop=0.2):
        super().__init__()
        self.backbone = backbone
        d = self.backbone.num_features
        self.norm = nn.LayerNorm(d)
        self.att_v = nn.Linear(d, att_dim)
        self.att_u = nn.Linear(d, att_dim)
        self.att_w = nn.Linear(att_dim, 1)
        self.drop = nn.Dropout(drop)
        self.head = nn.Linear(d, 1)
        nn.init.normal_(self.head.weight, std=0.01)
        nn.init.zeros_(self.head.bias)

    def encode(self, x, chunk=None):
        if chunk is None or x.shape[0] <= chunk:
            return self.backbone(x)
        return torch.cat([self.backbone(x[i:i + chunk]) for i in range(0, x.shape[0], chunk)])

    def pool(self, h, counts):
        # h: (N, D) windows of several studies concatenated; counts: windows per study. Returns (B,) logits.
        h = self.norm(h)
        a = self.att_w(torch.tanh(self.att_v(h)) * torch.sigmoid(self.att_u(h))).float()   # (N, 1)
        out, start = [], 0
        for n in counts:
            hs, wts = h[start:start + n].float(), torch.softmax(a[start:start + n], dim=0)
            pooled = (wts * hs).sum(0)
            out.append(self.head(self.drop(pooled)).squeeze(-1))
            start += n
        return torch.stack(out)


def build_backbone(cfg, load_weights=True):
    return timm.create_model(cfg['backbone'], pretrained=bool(cfg['pretrained'] and load_weights), num_classes=0)


def batch_tensors(batch, device, bucket=16):
    # The windows of a batch padded with zero images to a multiple of `bucket`: few distinct batch shapes (each new shape
    # costs about 10 s of cuDNN set-up on the local GPU). The padding is cut off before the pooling; the backbone has no
    # cross-sample layers (ConvNeXt: LayerNorm, per-sample drop path), so real windows are unaffected.
    x = np.concatenate([b['x'] for b in batch])
    n = len(x)
    if n % bucket:
        x = np.concatenate([x, np.zeros((bucket - n % bucket,) + x.shape[1:], x.dtype)])
    return torch.from_numpy(x).to(device, non_blocking=True), [len(b['x']) for b in batch], n


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def auc(y_bin, score):
    y_bin = np.asarray(y_bin)
    return float(roc_auc_score(y_bin, score)) if 0 < y_bin.sum() < len(y_bin) else float('nan')


@torch.no_grad()
def predict(model, ds, device, batch_studies, amp, num_workers=0):
    # Returns study ids in dataset order and float32 logits.
    model.eval()
    loader = DataLoader(ds, batch_size=batch_studies, shuffle=False, num_workers=num_workers, collate_fn=collate)
    logits, uids = [], []
    for batch in loader:
        x, counts, n = batch_tensors(batch, device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
            out = model.pool(model.encode(x, chunk=256)[:n], counts)
        logits.append(out.float().cpu().numpy())
        uids += [b['uid'] for b in batch]
    return uids, np.concatenate(logits)


# --------------------------------------------------------------------------- main
def main(CFG):
    t_start = time.time()
    tag, finding, fold, out = CFG['tag'], CFG['finding'], CFG['fold'], Path(CFG['out_dir'])
    out.mkdir(parents=True, exist_ok=True)
    stem = f'{CFG["version"]}_fold_{fold}_{tag}'             # checkpoints: v43_fold_<k>_<tag>_best.pt
    torch.manual_seed(CFG['seed'])
    device = torch.device(CFG['device'] if torch.cuda.is_available() or CFG['device'] == 'cpu' else 'cpu')
    amp = device.type == 'cuda'

    targets = pd.read_csv(CFG['targets'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
    fit = pd.read_csv(CFG['train_targets'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
    folds = pd.read_csv(CFG['folds'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
    rows = pd.read_csv(CFG['rows'], dtype={'StudyInstanceUID': str, 'shard': str}).set_index('StudyInstanceUID')
    fold_of = folds.fold.reindex(targets.index)
    usable = set(rows.index)
    gold_all = set(targets.index[targets.is_gold.astype(bool)])
    assert not (set(fit.index) & gold_all), 'gold studies are never trained or validated on'
    train_ids = [u for u in fit.index if fold_of[u] != fold and u in usable]
    val_ids = [u for u in fit.index if fold_of[u] == fold and u in usable]
    gold_ids = [u for u in targets.index if u in gold_all and u in usable]
    y_fit = fit[finding].astype(np.float32)
    y_soft = targets[finding].astype(np.float32)
    w_all = targets[f'{finding}__weight'].astype(np.float32)
    assert np.isfinite(y_fit.loc[train_ids + val_ids]).all() and (w_all > 0).all()
    log(f'{tag} ({finding}) fold {fold}: train {len(train_ids)}, val {len(val_ids)}, gold {len(gold_ids)}, device {device}')

    def make_ds(ids, y, k, train):
        r = [(u, rows.at[u, 'shard'], int(rows.at[u, 'start']), int(rows.at[u, 'n'])) for u in ids]
        return RoiDataset(r, y.loc[ids].to_numpy(np.float32), w_all.loc[ids].to_numpy(np.float32), CFG['crops_dir'], k,
                          train, CFG['seed'])

    train_ds = make_ds(train_ids, y_fit, CFG['k_train'], True)
    model = Net(build_backbone(CFG)).to(device)
    mb = CFG['micro_batch_studies']
    probe = {'micro_batch_studies': mb, 'grad_checkpointing': False, 'attempts': []}
    if device.type == 'cuda':
        while True:
            n_tok = mb * CFG['k_train']
            try:
                xx = torch.randn(n_tok, 3, CFG['img_h'], CFG['img_w'], device=device)
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    lo = model.pool(model.encode(xx), [CFG['k_train']] * mb)
                lo.float().sum().backward()
                model.zero_grad(set_to_none=True)
                probe['attempts'].append({'tokens': n_tok, 'ok': True,
                                          'peak_reserved_gb': round(torch.cuda.max_memory_reserved() / 1e9, 2)})
                del xx, lo
                torch.cuda.empty_cache()
                break
            except torch.OutOfMemoryError:
                model.zero_grad(set_to_none=True)
                torch.cuda.empty_cache()
                probe['attempts'].append({'tokens': n_tok, 'ok': False})
                if not probe['grad_checkpointing']:
                    model.backbone.set_grad_checkpointing(True)
                    probe['grad_checkpointing'] = True
                elif mb > 1:
                    mb = mb // 2
                    probe['micro_batch_studies'] = mb
                else:
                    raise
        log(f'{tag}: memory probe {probe}')
    accum = max(1, CFG['effective_batch_studies'] // mb)
    head_params = [p for n, p in model.named_parameters() if not n.startswith('backbone.')]
    opt = torch.optim.AdamW([{'params': list(model.backbone.parameters()), 'lr': CFG['lr_backbone']},
                             {'params': head_params, 'lr': CFG['lr_head']}], weight_decay=CFG['weight_decay'])
    base_lrs = [CFG['lr_backbone'], CFG['lr_head']]
    scaler = torch.amp.GradScaler('cuda', enabled=amp)
    steps_per_epoch = math.ceil(len(train_ids) // mb / accum)
    total_steps = steps_per_epoch * CFG['epochs']
    warm = steps_per_epoch * CFG['warmup_epochs']

    def set_lr(step):
        f = step / max(1, warm) if step < warm else 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total_steps - warm)))
        for g, lr in zip(opt.param_groups, base_lrs):
            g['lr'] = lr * f

    receipt = {'version': CFG['version'], 'tag': tag, 'finding': finding, 'fold': fold, 'status': 'running',
               'cfg': {k: v for k, v in CFG.items() if k not in ('targets', 'train_targets', 'folds', 'rows', 'crops_dir', 'out_dir')},
               'n_train': len(train_ids), 'n_val': len(val_ids), 'n_gold': len(gold_ids),
               'train_pos_rate_soft05': float((y_soft.loc[train_ids] >= 0.5).mean()),
               'val_pos_rate_soft05': float((y_soft.loc[val_ids] >= 0.5).mean()),
               'train_targets_sha256': hashlib.sha256(Path(CFG['train_targets']).read_bytes()).hexdigest(),
               'rows_sha256': hashlib.sha256(Path(CFG['rows']).read_bytes()).hexdigest(),
               'memory_probe': probe, 'micro_batch_studies': mb, 'accum': accum,
               'backbone_params': int(sum(p.numel() for p in model.backbone.parameters())),
               'device_name': torch.cuda.get_device_name(0) if amp else 'cpu',
               'torch': torch.__version__, 'timm': timm.__version__, 'epoch_eval': {}}
    write = lambda: (out / f'{CFG["version"]}_receipt_fold{fold}_{tag}.json').write_text(json.dumps(receipt, indent=2))

    def run_eval(ids, y):
        ds = make_ds(ids, y, CFG['k_infer'], False)
        t = time.time()
        uids, z = predict(model, ds, device, CFG['eval_batch_studies'], amp, CFG['num_workers'])
        assert uids == list(ids)
        return z, time.time() - t

    step = micro = n_images = 0
    best, best_epoch, stopped = -1.0, -1, False
    t_data = t_compute = 0.0
    if amp:
        torch.cuda.reset_peak_memory_stats()
    for epoch in range(CFG['epochs']):
        train_ds.epoch = epoch
        loader = DataLoader(train_ds, batch_size=mb, shuffle=True, num_workers=CFG['num_workers'], collate_fn=collate,
                            drop_last=True, persistent_workers=False, generator=torch.Generator().manual_seed(CFG['seed'] + epoch))
        model.train()
        opt.zero_grad(set_to_none=True)
        loss_sum, loss_n = 0.0, 0
        t0 = time.time()
        for batch in loader:
            t1 = time.time()
            t_data += t1 - t0
            x, counts, n = batch_tensors(batch, device)
            y = torch.from_numpy(np.stack([b['y'] for b in batch])).to(device)
            w = torch.from_numpy(np.stack([b['w'] for b in batch])).to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                logits = model.pool(model.encode(x)[:n], counts)
            loss = (F.binary_cross_entropy_with_logits(logits.float(), y, reduction='none') * w).sum() / w.sum()
            scaler.scale(loss / accum).backward()
            micro += 1
            n_images += n
            loss_sum += loss.item()
            loss_n += 1
            if micro % accum == 0:
                set_lr(step)
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                step += 1
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t0 = time.time()
            t_compute += t0 - t1
            if micro % 50 == 0:
                log(f'{tag} epoch {epoch} micro {micro} loss {loss.item():.4f} '
                    f'img/s {n_images / max(t_data + t_compute, 1e-9):.1f} data share {t_data / max(t_data + t_compute, 1e-9):.2f}')
            if time.time() - t_start > CFG['time_limit_s']:
                stopped = True
                break
        if stopped:
            receipt.update(status='stopped_early', stopped_epoch=epoch)
            break
        rec = {'train_batch_loss_mean': loss_sum / max(loss_n, 1), 'elapsed_s': round(time.time() - t_start, 1)}
        if epoch + 1 >= CFG['eval_from_epoch']:
            z_val, s_val = run_eval(val_ids, y_fit)
            p_val = sigmoid(z_val)
            rec.update(val_auc=auc(y_soft.loc[val_ids] >= 0.5, p_val), val_seconds=round(s_val, 1),
                       val_loss=float(F.binary_cross_entropy_with_logits(torch.from_numpy(z_val), torch.from_numpy(
                           y_fit.loc[val_ids].to_numpy(np.float32)))))
            if rec['val_auc'] > best:
                best, best_epoch = rec['val_auc'], epoch
                torch.save({'model': model.state_dict(), 'epoch': epoch, 'cfg': CFG}, out / f'{stem}_best.pt')
                pd.DataFrame({finding: p_val}, index=pd.Index(val_ids, name='StudyInstanceUID')).to_csv(
                    out / f'{CFG["version"]}_oof_fold{fold}_{tag}.csv')
        receipt['epoch_eval'][str(epoch)] = rec
        log(f'{tag} epoch {epoch}: val AUC {rec.get("val_auc", float("nan")):.4f} loss {rec.get("val_loss", float("nan")):.4f} '
            f'| batch loss {rec["train_batch_loss_mean"]:.4f} | best {best:.4f} (epoch {best_epoch})')
        write()
        if best_epoch >= 0 and epoch - best_epoch >= CFG['patience']:
            receipt['early_stopped_after_epoch'] = epoch
            break

    elapsed = t_data + t_compute
    receipt.update(train_images=n_images, train_seconds=round(elapsed, 1),
                   train_images_per_s=round(n_images / max(elapsed, 1e-9), 1),
                   data_share=round(t_data / max(elapsed, 1e-9), 3), micro_steps=micro, optimizer_steps=step)
    if amp:
        receipt.update(peak_reserved_gb=round(torch.cuda.max_memory_reserved() / 1e9, 2))
    if best_epoch < 0:
        receipt.update(status='no_valid_eval' if receipt['status'] == 'running' else receipt['status'],
                       total_seconds=round(time.time() - t_start, 1))
        write()
        log(json.dumps({k_: v for k_, v in receipt.items() if k_ != 'epoch_eval'}))
        return
    # Gold evaluation of the best checkpoint (descriptive only).
    model.load_state_dict(torch.load(out / f'{stem}_best.pt', map_location=device)['model'])
    z_gold, _ = run_eval(gold_ids, y_soft)
    p_gold = sigmoid(z_gold)
    pd.DataFrame({finding: p_gold}, index=pd.Index(gold_ids, name='StudyInstanceUID')).to_csv(
        out / f'{CFG["version"]}_gold_fold{fold}_{tag}.csv')
    receipt.update(status='done' if receipt['status'] == 'running' else 'stopped_early_evaluated', best_epoch=best_epoch,
                   val_auc=best, gold_auc=auc(targets.loc[gold_ids, f'{finding}__gold'].to_numpy() > 0.5, p_gold),
                   total_seconds=round(time.time() - t_start, 1))
    write()
    log(json.dumps({k_: v for k_, v in receipt.items() if k_ not in ('epoch_eval', 'cfg')}))


if __name__ == '__main__':
    main(json.loads(sys.argv[1]))


## Train the experts

One process per expert, one after the other. A stale receipt is removed before a launch and never counts.

In [ ]:
results = {}
log_path = WORK / f'{VERSION}_log.txt'
for tag, finding, comp in EXPERTS:
    out_dir = WORK / f'fold{FOLD}_{tag}'
    receipt_file = out_dir / f'{VERSION}_receipt_fold{FOLD}_{tag}.json'
    if receipt_file.exists():
        receipt_file.unlink()
    cfg = {**CFG, 'tag': tag, 'finding': finding, 'rows': str(WORK / f'v43_rows_{tag}.csv'), 'crops_dir': str(ROI_DIR / 'crops'),
           'targets': PATHS['targets'], 'train_targets': PATHS['train_targets'], 'folds': PATHS['folds'],
           'out_dir': str(out_dir), 'time_limit_s': TIME_LIMIT_S}
    t = time.time()
    with open(log_path, 'a', encoding='utf-8') as logf:
        rc = subprocess.run([sys.executable, '-u', str(WORK.parent / 'v43_train.py'), json.dumps(cfg)], stdout=logf,
                            stderr=subprocess.STDOUT, timeout=TIME_LIMIT_S + 1800).returncode
    rec = json.loads(receipt_file.read_text()) if receipt_file.exists() else {'status': 'missing'}
    results[tag] = {'exit_code': rc, 'wall_s': round(time.time() - t, 1),
                    **{k: rec.get(k) for k in ('status', 'best_epoch', 'val_auc', 'gold_auc', 'n_train', 'n_val', 'n_gold',
                                               'train_images_per_s', 'peak_reserved_gb', 'total_seconds')}}
    print(tag, json.dumps(results[tag]), flush=True)
(WORK / f'{VERSION}_experts_summary.json').write_text(json.dumps(results, indent=2))


## Diagnostics (no gate, no selection)

`order_only_blend` is the v44 rule of design section 5: inside the valid rows, the order of `0.5 rank(incumbent) +
0.5 rank(expert)` (ties by the incumbent's rank) receives the incumbent's own sorted values; every other row keeps its value.
Here arm N's fold-0 OOF stands in for the incumbent. Fold-0 numbers are leaky (see the header).

In [ ]:
from sklearn.metrics import roc_auc_score

def order_only_blend(inc, exp, valid, w=0.5):
    inc = np.asarray(inc, np.float64)
    out = inc.copy()
    idx = np.flatnonzero(np.asarray(valid, bool) & np.isfinite(np.asarray(exp, np.float64)))
    if len(idx) < 2:
        return out
    a = pd.Series(inc[idx]).rank(method='average').to_numpy()
    e = pd.Series(np.asarray(exp, np.float64)[idx]).rank(method='average').to_numpy()
    order = np.lexsort((a, (1.0 - w) * a + w * e))
    out[idx[order]] = np.sort(inc[idx])
    return out

_rng = np.random.default_rng(0)
_inc, _exp = _rng.random(50), _rng.random(50)
_val = _rng.random(50) < 0.7
_b = order_only_blend(_inc, _exp, _val)
assert np.array_equal(np.sort(_b), np.sort(_inc)) and np.array_equal(_b[~_val], _inc[~_val])
assert np.array_equal(order_only_blend(_inc, _inc, _val), _inc) and np.array_equal(order_only_blend(_inc, _exp, ~np.ones(50, bool)), _inc)

def auc_(y, s):
    y = np.asarray(y, bool)
    return float(roc_auc_score(y, s)) if 0 < y.sum() < len(y) else float('nan')

def paired_boot(y, a, b, n=2000, seed=0):
    rng, d = np.random.default_rng(seed), []
    y = np.asarray(y, bool)
    for _ in range(n):
        s = rng.integers(0, len(y), len(y))
        if 0 < y[s].sum() < len(y):
            d.append(auc_(y[s], b[s]) - auc_(y[s], a[s]))
    d = np.asarray(d)
    return {'mean': float(d.mean()), 'ci95': [float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))],
            'p_nonpos': float((d <= 0).mean())}

n_gold = pd.read_csv(N_GOLD_PATH, dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID') if N_GOLD_PATH else None
diagnostics = {}
for tag, finding, comp in EXPERTS:
    out_dir = WORK / f'fold{FOLD}_{tag}'
    f_oof = out_dir / f'{VERSION}_oof_fold{FOLD}_{tag}.csv'
    if not f_oof.exists():
        diagnostics[tag] = {'status': results.get(tag, {}).get('status', 'missing')}
        continue
    ex = pd.read_csv(f_oof, dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')[finding]
    va = [u for u in pool if fold_of[u] == FOLD]
    y = (soft.loc[va, finding] >= 0.5).to_numpy()
    n_va = n_oof.loc[va, finding].to_numpy()
    use = np.array([u in ex.index for u in va])
    e_va = ex.reindex(va).to_numpy()
    rm = (pd.Series(n_va[use]).rank().to_numpy() + pd.Series(e_va[use]).rank().to_numpy()) / 2
    blend = order_only_blend(n_va, e_va, use)
    d = {'val_usable': {'n': int(use.sum()), 'pos': int(y[use].sum()), 'expert': auc_(y[use], e_va[use]),
                        'n_oof': auc_(y[use], n_va[use]), 'rank_mean': auc_(y[use], rm)},
         'val_all': {'n': len(va), 'n_oof': auc_(y, n_va), 'order_only_blend': auc_(y, blend),
                     'blend_minus_n': paired_boot(y, n_va, blend)}}
    f_gold = out_dir / f'{VERSION}_gold_fold{FOLD}_{tag}.csv'
    if n_gold is not None and f_gold.exists():
        eg = pd.read_csv(f_gold, dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')[finding]
        g_ids = list(n_gold.index)
        yg = (targets.loc[g_ids, f'{finding}__gold'].to_numpy() > 0.5)
        ng = n_gold.loc[g_ids, finding].to_numpy()
        ug = np.array([u in eg.index for u in g_ids])
        egv = eg.reindex(g_ids).to_numpy()
        d['gold'] = {'n': len(g_ids), 'usable': int(ug.sum()), 'pos': int(yg.sum()),
                     'expert_usable': auc_(yg[ug], egv[ug]), 'n_oof_usable': auc_(yg[ug], ng[ug]),
                     'n_all': auc_(yg, ng), 'order_only_blend_all': auc_(yg, order_only_blend(ng, egv, ug))}
    diagnostics[tag] = d
    print(tag, json.dumps(d, indent=1), flush=True)
(WORK / f'{VERSION}_diagnostics.json').write_text(json.dumps(diagnostics, indent=2))
